# XIX-Upscaler — Video (Colab Experimental)

Notebook **0.2.0** untuk pekerjaan dari desktop XIX-Upscaler. Pilih video dan Folder Hasil di aplikasi, tekan **Start**, lalu tunggu upload selesai. Di notebook ini pilih **Runtime → Run all** dan izinkan mount Drive dengan akun yang sama seperti di desktop. Notebook memeriksa GPU dan folder desktop, memasang worker yang terverifikasi, lalu memproses antrean video. Tidak ada pengaturan yang perlu diubah di notebook.

> Batas setiap video: **100 MB (104.857.600 byte)** dan **60 detik**. Fitur eksperimental ini memakai Drive, GPU, dan kuota Colab milik pengguna. Gunakan runtime GPU dan biarkan tab terbuka. Jika sesi terputus, jalankan **Run all** lagi untuk melanjutkan dari checkpoint. Desktop mengunduh dan memverifikasi hasil ke Folder Hasil.

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys
import urllib.request
import torch

assert torch.cuda.is_available(), "GPU tidak tersedia. Pilih Runtime → Change runtime type → GPU."
print(f"GPU: {torch.cuda.get_device_name(0)}")

WORKER_VERSION = "0.2.0"
RELEASE_COMMIT = "0c8647e5b437cd3cbc3cbe5723971afb701fb20f"
WHEEL_NAME = "xix_colab_worker-0.2.0-py3-none-any.whl"
WHEEL_URL = f"https://raw.githubusercontent.com/mfahryf/xix-upscaler-colab/{RELEASE_COMMIT}/dist/{WHEEL_NAME}"
WHEEL_SIZE = 38911
WHEEL_SHA256 = "f2eb63c30f6639874ee1771d154b72480609b4664f49f18c271bbaa22eeee117"
WHEEL_PATH = Path("/content") / WHEEL_NAME
JOBS_DIR = Path("/content/drive/MyDrive/XIX-Upscaler/jobs")
CACHE_DIR = Path("/content/xix-upscaler-cache")

In [ ]:
from google.colab import drive
import json
import uuid

drive.mount("/content/drive")
MARKER_PATH = JOBS_DIR.parent / "desktop-marker.json"
if not MARKER_PATH.is_file():
    raise RuntimeError("Akun Drive tidak cocok atau belum ada job desktop. Gunakan akun yang sama dengan aplikasi XIX-Upscaler, lalu tekan Start di desktop dan Run all kembali.")

def marker_object(pairs):
    value = dict(pairs)
    if len(value) != len(pairs):
        raise ValueError("Field marker berulang")
    return value

try:
    marker = json.loads(MARKER_PATH.read_text(encoding="utf-8"), object_pairs_hook=marker_object)
    if not isinstance(marker, dict) or set(marker) != {"schema_version", "desktop_install_id"}:
        raise ValueError("Format marker tidak cocok")
    if type(marker["schema_version"]) is not int or marker["schema_version"] != 1:
        raise ValueError("Versi marker tidak cocok")
    if not isinstance(marker["desktop_install_id"], str):
        raise ValueError("Identitas instalasi tidak valid")
    uuid.UUID(marker["desktop_install_id"])
except (OSError, ValueError):
    raise RuntimeError("Marker desktop tidak valid. Buka notebook dari aplikasi XIX-Upscaler yang sesuai, gunakan akun Drive yang sama, lalu tekan Start dan Run all kembali.") from None
if not JOBS_DIR.is_dir():
    raise RuntimeError("Folder jobs desktop belum tersedia. Tekan Start di aplikasi XIX-Upscaler, tunggu upload selesai, lalu Run all kembali.")
print(f"Folder antrean siap: {JOBS_DIR}")

In [ ]:
with urllib.request.urlopen(WHEEL_URL, timeout=60) as response:
    wheel_bytes = response.read()
if len(wheel_bytes) != WHEEL_SIZE:
    raise RuntimeError("Ukuran paket worker tidak cocok")
if hashlib.sha256(wheel_bytes).hexdigest() != WHEEL_SHA256:
    raise RuntimeError("SHA-256 paket worker tidak cocok")
WHEEL_PATH.write_bytes(wheel_bytes)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(WHEEL_PATH)],
    check=True,
)
print(f"Worker XIX {WORKER_VERSION} terpasang dan terverifikasi")

In [ ]:
subprocess.run(
    [sys.executable, "-m", "xix_colab_worker", "verify-environment", "--jobs-dir", str(JOBS_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "xix_colab_worker", "verify-locks", "--cache-dir", str(CACHE_DIR)],
    check=True,
)

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "xix_colab_worker",
        "run-queue",
        "--jobs-dir",
        str(JOBS_DIR),
        "--cache-dir",
        str(CACHE_DIR),
    ],
    check=False,
)
if result.returncode not in (0, 1):
    raise RuntimeError(f"Worker berhenti dengan kode {result.returncode}")
if result.returncode == 1:
    print("Ada job yang gagal. Periksa ringkasan di atas dan alasan kegagalan di desktop.")
else:
    print("Putaran antrean selesai. Periksa ringkasan di atas dan status di desktop; job dapat selesai, dijeda, dibatalkan, atau dilewati.")
print("Desktop mengunduh dan memverifikasi hasil job yang selesai ke Folder Hasil.")